### Vector Search

Keyword search in rag is when we only retain the meaningful parts of user query and look for those in the documents.Keyword search looks for exact matches, so if the wording is different, many relevant documents may be missed. Vector search in rag is when we use embeddings to find documents that are semantically similar to the user query.

Before we can do vector search, we need to turn our text into vectors. We call this process embedding: we embed text into a vector space. The vectors we get back are also called "embeddings."
Embedding can be done to both word and sentences.

We put all our docs in a vector space, so when a user inserts his question we embed it into our vector space. The 
neighboring points are similar questions to our user's. Which mean that the surrounding points are our search results.

In [1]:
# Importing our sentence transformer model
from sentence_transformers import SentenceTransformer

# Creating a model object
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
# Encoding our documents into vectors
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
# v1 is a vector, an array of 384 numbers.
v1

array([ 2.13903747e-02, -7.39799738e-02,  1.42071163e-03,  2.13816650e-02,
        2.45113466e-02,  3.15583199e-02, -1.10839762e-01, -1.05017520e-01,
       -6.18259162e-02, -6.42313948e-03,  3.72393779e-03,  9.06393379e-02,
       -9.49937943e-03,  6.53977245e-02,  1.10947024e-02, -2.10097581e-02,
       -3.35125178e-02, -4.31677699e-02,  9.96347424e-03,  1.41970078e-02,
       -6.40415698e-02, -7.04176398e-03, -7.91187808e-02,  5.80030866e-02,
        1.30211480e-03,  4.19730321e-03,  5.70978969e-02,  6.39448017e-02,
        2.49903016e-02, -3.95876318e-02, -3.79506499e-02,  2.70394702e-02,
        1.79423690e-02,  1.72272492e-02,  3.43311355e-02,  9.29057878e-03,
        5.86055033e-02, -4.97789793e-02, -5.05371578e-03,  4.34328988e-02,
       -1.56622995e-02, -2.97534633e-02, -5.13325585e-03,  5.13414964e-02,
        6.16066018e-03,  6.86980560e-02, -1.29505433e-02, -5.61938323e-02,
       -1.08264983e-02,  5.96683770e-02,  5.29939868e-02, -3.42755094e-02,
       -4.15273942e-02, -

In [3]:
v1.shape

(384,)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
similarity = v1.dot(dv)

In [6]:
similarity

np.float32(0.32332402)

In [7]:
similarities = model.similarity(v1, [dv])
print(similarities)

tensor([[0.3233]])


/Users/zineb/llm-zoomcamp-2026-code/.venv/lib/python3.12/site-packages/sentence_transformers/util/tensor.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  a = torch.tensor(a)


In [8]:
from ingest import load_faq_data

In [9]:
documents = load_faq_data()

In [10]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [11]:
# We need to first extract the text from the documents.

texts = []

for doc in documents:
    text =doc['question'] + " " + doc['answer']
    texts.append(text)

In [12]:
texts[10]

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [13]:
len(texts)

1350

We have 1350 text that we need to embed. That will take a long time so we will do that in batches.

In [14]:
# First we import tqdm so we can watch the progress:
from tqdm.auto import tqdm

In [15]:
# Next we chunk the dataset into batches of 50 and encode each batch:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)


  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [16]:
vectors[10].shape

(384,)

We turn them into a 2-dimensional array (matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

In [17]:
import numpy as np
X = np.array(vectors)

In [18]:
X.shape

(1350, 384)

In [19]:
scores = X.dot(v1)
scores

array([ 0.48740596,  0.20991942,  0.7629412 , ..., -0.08637968,
        0.03759797, -0.03037032], shape=(1350,), dtype=float32)

In [20]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629412))

In [23]:
documents[2]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [24]:
top5 = np.argsort(scores)[-5:]
top5

array([  7, 538, 907, 625,   2])

In [25]:
scores[top5]

array([0.5601001, 0.6536312, 0.7192135, 0.7579372, 0.7629412],
      dtype=float32)

In [26]:
#reverse the order of the top5 indices to get the highest scores first
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [27]:
print("Top 5 most similar questions:")
for i in top5:
    print(f"Score: {scores[i]:.4f}, Question: {documents[i]['question']}")
    print(f"Answer: {documents[i]['answer']}\n")
    


Top 5 most similar questions:
Score: 0.7629, Question: Course: Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homework.

Be aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.

Score: 0.7579, Question: Course - Can I still join the course after the start date?
Answer: Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.

Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.

Score: 0.7192, Question: The course has already started. Can I still join it?
Answer: Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.

In order to get a certif

In [29]:
for i in top5:
    print(f"Score: {scores[i]}")
    print(f"Document: {documents[i]}")

Score: 0.7629411816596985
Document: {'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}
Score: 0.7579371929168701
Document: {'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}
Score: 0.7192134857177734
Document: {'id': '41aabbd7c5

In [52]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [53]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [55]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [56]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co